# **Utilisation des modèles de plongement des mots et de documents**

In [1]:
import numpy as np
import pandas as pd
import nltk
import gensim
from nltk.stem import WordNetLemmatizer
from string import punctuation
from nltk.corpus import stopwords
import matplotlib.pyplot as plt


In [ ]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
print(np.__version__)
print(pd.__version__)
print(nltk.__version__)
print(gensim.__version__)

# Etude d'un corpus de documents. Dans cette exemple le corpus est un paragraphe où chaque phrase est considérée comme étant un document.


In [4]:
# Exemple: Définition de notre corpus à partir du texte du discours d'Obama
# with open("Discours_Barack_Obama.txt", "r") as fichier:
#	     Dis_Obama=fichier.read()

# Ce corpus est compsé de 4 documents(phrases)
Dis_Obama ="If there is anyone out there who still doubts that America is a place where all things are possible; who still wonders if the dream of our founders is alive in our time; who still questions the power of our democracy, tonight is your answer. It's the answer told by lines that stretched around schools and churches in numbers this nation has never seen; by people who waited three hours and four hours, many for the very first time in their lives, because they believed that this time must be different; that their voice could be that difference. It's the answer spoken by young and old, rich and poor, Democrat and Republican, black, white, Latino, Asian, Native American, gay, straight, disabled and not disabled - Americans who sent a message to the world that we have never been a collection of red states and blue states; we are, and always will be, the United States of America. It's the answer that led those who have been told for so long by so many to be cynical, and fearful, and doubtful of what we can achieve to put their hands on the arc of history and bend it once more toward the hope of a better day."

# On peut faire le prétraitement sur le texte initial avant la détection des pharses et des mots ou après.

## Représenter le corpus par une liste de documents (Décomposer le document en liste de phrases)

In [ ]:
# Segmentation des phrases avec nltk
from nltk.tokenize import sent_tokenize

corpus_in = sent_tokenize(Dis_Obama)

print(corpus_in)
print(len(corpus_in), "documents")

## Prétraitement des documents (phrases) pour la préparation du corpus

In [ ]:
# Word tokenizer
from nltk.tokenize import word_tokenize
lem = WordNetLemmatizer()

# Segmenter le corpus en gardant la structure des phrases
corpus = [word_tokenize(e) for e in corpus_in]
print("Sous forme d'une liste de listes de mots: \n", corpus)
print(len(sum(corpus,[])),"mots")

# Convertir les mots en minuscules
corpus = [[mot.lower() for mot in sent] for sent in corpus]
print("Sous forme d'une liste de listes de mots en minuscules: \n",corpus)
print(len(sum(corpus,[])),"mots")

# Supprimer les symboles de ponctuation
corpus = [[mot for mot in sent if mot not in punctuation] for sent in corpus]
print("Sous forme d'une liste de listes de mots sans ponctuation: \n",corpus)
print(len(sum(corpus,[])),"mots")

# Représenter les mots par leur forme canonique
corpus = [[lem.lemmatize(mot) for mot in sent] for sent in corpus]
print("Sous forme d'une liste de listes de forme canonique de mot: \n",corpus)
print(len(sum(corpus,[])),"mots")

#Supprimer les mots vides
mots_vides = stopwords.words('english')
corpus_without_sw = [[mot for mot in sent if mot not in mots_vides] for sent in corpus]
print("Sous forme d'une liste de listes de mots sans les mots vides: \n",corpus_without_sw)
print(len(sum(corpus_without_sw,[])),"mots")

#Plongement des mots avec un modèle Word2Vec
Pour voir d'autres fonctions ou paramètres:
https://radimrehurek.com/gensim/models/word2vec.html

# Entrainer un modèle Word2Vec avec notre corpus en gardant les mots vides


In [ ]:
# Création d'un modèle de plongement word2vec en utilisant le corpus des formes canoniques

from gensim.models import Word2Vec

# Initialisez le modèle avec les paramètres souhaités
# Dans cet exemple, on projette un mot vers un vecteur de 2 dimensions
# Ce choix nous permettra d'afficher les mots dans un plan pour voir les distances entre mots.
modele1 = Word2Vec(vector_size=2, window=5, min_count=1)

# Construisez le vocabulaire à partir du corpus
modele1.build_vocab(corpus)

# Entraînez le modèle
modele1.train(corpus, total_examples=modele1.corpus_count, epochs=10)

print("Nombre de mots du Vocabulaire (mots uniques du corpus), taille du vecteur : ", modele1.wv.vectors.shape)
print("Nombre de mots dans le corpus : ", modele1.corpus_total_words)
print("Nombre de documents : ", modele1.corpus_count)


In [ ]:
print("La taille du Vocabulaire: (mots uniques)", len(modele1.wv.key_to_index))
print("Vocabulaire:", modele1.wv.key_to_index)
print("Vecteur pour 'answer':", modele1.wv['answer'])

In [ ]:
# On peut initialiser, construire le vocabulaire et entraîner le modèle au même temps

#modele1 = Word2Vec(corpus,vector_size=10,window=5,min_count=1)

In [ ]:
# Affichage des mots du vocabulaire sur un plan 2D
import matplotlib.pyplot as plt

mots=list(modele1.wv.key_to_index.keys())
X=[modele1.wv[x][0] for x in mots ]
Y=[modele1.wv[x][1] for x in mots ]
plt.scatter(X, Y)
for i in range(len(X)):
    plt.annotate(mots[i],(X[i],Y[i]))
plt.show()

# Entrainer un modèle Word2Vec avec notre corpus en supprimant les mots vides

In [ ]:
# Création d'un modèle de plongement word2vec en utilisant le corpus des formes canoniques après suppression des mots vides
from gensim.models import Word2Vec

# Initialisez le modèle avec les paramètres souhaités
modele = Word2Vec(vector_size=2, window=5, min_count=1)

# Construisez le vocabulaire à partir du corpus
modele.build_vocab(corpus_without_sw)

# Entraînez le modèle
modele.train(corpus_without_sw, total_examples=modele.corpus_count, epochs=10)

print("Nombre de mots du Vocabulaire (mots uniques du corpus), taille du vecteur : ", modele.wv.vectors.shape)
print("Nombre de mots dans le corpus : ", modele.corpus_total_words)
print("Nombre de documents : ", modele.corpus_count)

In [ ]:
# On peut initialiser, construire le vocabulaire et entraîner le modèle au même temps
#modele = Word2Vec(corpus_without_sw,vector_size=2,window=5,min_count=1)

In [ ]:
# Affichez les mots du vocabulaire et leurs vecteurs issus du modèle Word2Vec
print("La taille du Vocabulaire: (mots uniques)", len(modele.wv.key_to_index))
print("Vocabulaire:", modele.wv.key_to_index)
print("Vecteur pour 'answer':", modele.wv['answer'])

In [ ]:
print(type(modele))

In [ ]:
dir(modele)

In [ ]:
#l'objet wv comporte les vecteurs des mots présents dans le vocabulaire.
# ainsi que des fonctions permettant d'accéder à ces vecteurs ou d'effectuer des calculs, comme la similarité entre mots.
lesmots = modele.wv

print(type(lesmots))

In [ ]:
dir(lesmots)

In [ ]:
# Afiicher le dictionnaire du vocabulaire
lesmots.key_to_index

In [ ]:
# Afficher la taille de la matrice des vecteurs (nb de mots, taille de chaque vecteur de mot)
lesmots.vectors.shape

In [ ]:
# Exemples de vecteurs de mots
vec1 = lesmots['doubt']
print(vec1)

vec2 = lesmots['wonder']
print(vec2)

In [ ]:
# Exemple de calcul de la distance cosinus entre les représentations de deux mots
print(np.dot(vec1,vec2)/(np.linalg.norm(vec1)*np.linalg.norm(vec2)))
lesmots.similarity('doubt','wonder')

In [ ]:
# Chercher les mots les plus similaires à un mot requête.
lesmots.most_similar("doubt")

In [ ]:
# Chercher les mots les plus similaires à une liste de mots.
print(lesmots.most_similar(positive=['wonder','doubt'],topn=3))

In [ ]:
# Chercher les mots les plus similaires à un mot et différents d'un autre.
print(lesmots.most_similar(positive=['wonder'],negative=['doubt'],topn=3))

In [ ]:
# Chercher l'intrus dans une liste
print(lesmots.doesnt_match(['wonder','better','alive','doubt']))

In [ ]:
# Affichage des mots du vocabulaire sur un plan 2D


mots=list(lesmots.key_to_index.keys())
X=[lesmots[x][0] for x in mots ]
Y=[lesmots[x][1] for x in mots ]
plt.scatter(X, Y)
for i in range(len(X)):
    plt.annotate(mots[i],(X[i],Y[i]))
plt.show()

# Utiliser des modèles Word2Vec pré-entainés


In [ ]:
# Charger le modèle pré entrainé à partir de https://wikipedia2vec.github.io/wikipedia2vec/pretrained/
from gensim.models import keyedvectors
Plon_mots = keyedvectors.load_word2vec_format("http://wikipedia2vec.s3.amazonaws.com/models/en/2018-04-20/enwiki_20180420_100d.txt.bz2",binary=False,unicode_errors='ignore')

In [ ]:
print(type(Plon_mots))

In [ ]:
# Affichage de la taille du corpus utilisé pour apprendre le modèle utilisé ainsi que la taille du vecteur latent pour représenter les mots
print(Plon_mots.vectors.shape)

In [ ]:
# Exemple du vecteur caractéristique d'un mot présent dans le corpus initial utilisé pour l'apprentissage du modèle
Plon_mots['table']

In [ ]:
# Exemple d'une exception levée pour un mot non présent dans le corpus initial utilisé pour l'apprentissage du modèle
Plon_mots['USA']

## Représenter un document par la moyenne des mots qu'il comporte et qui sont présents dans le vocabulaire du corpus utilisé pour l'apprentissage du modèle pré entrainé.

In [ ]:
def doc_as_words_mean(doc,Plon_mots):
    """
    Cette fonction calcule la moyenne des vecteurs des mots d'un document (phrase)
    """

    tailleVec = Plon_mots.vectors.shape[1]

    vec_doc = np.zeros(tailleVec)
    nb_tokens = 0.0

    for token in doc:
      # Si le mot du document n'appartient au vocabulaire du modèle pré entainé, il n'est pas pris en considération
        try:
            vec_token = Plon_mots[token]
            vec_doc = vec_doc + vec_token
            nb_tokens +=  1
        except:
            pass


    if (nb_tokens > 0.0):
        vec_doc = vec_doc/nb_tokens

    return vec_doc

In [ ]:
# Calcul des vecteurs caractéristiques des 4 phrases de notre paragraphe
Vec_Docs = []
for doc in corpus:
    vec = doc_as_words_mean(doc,Plon_mots)
    Vec_Docs.append(vec)

matVec = np.array(Vec_Docs)
print(matVec.shape)
print(matVec)


In [ ]:
#D'autres modèles Word2Vec pré-entrainés à partir de ce site:  https://fauconnier.github.io/
# Dans l'exemple d'avant on a testé un modèle en format texte, ici on peut tester un modèle en bianire
Plon_mots_1 = keyedvectors.load_word2vec_format("https://embeddings.net/embeddings/frWac_non_lem_no_postag_no_phrase_200_cbow_cut0.bin",binary=True,unicode_errors='ignore')
#taille du dictionnaire
print(len(Plon_mots_1.key_to_index))

In [ ]:
# calculer les vecteurs caractéristiques des phrases en moyennant les vecteurs Word2vec des mots présents dans chaque phrase

Vec_Docs = []
for doc in corpus:
    vec = doc_as_words_mean(doc,Plon_mots_1)
    Vec_Docs.append(vec)

matVec = np.array(Vec_Docs)
print(matVec.shape)
print(matVec)

# Plongement des mots avec un modèle FastText
Pour voir d'autres fonctions ou paramètres : https://radimrehurek.com/gensim/models/fasttext.html

# Entrainer un modèle FastText avec notre corpus en gardant les mots vides

In [ ]:
from gensim.models import FastText

# Initialisez le modèle avec les paramètres souhaités
modele1 = FastText(vector_size=2, window=5, min_count=1)

# Construisez le vocabulaire à partir du corpus
modele1.build_vocab(corpus)

# Entraînez le modèle
modele1.train(corpus, total_examples=modele1.corpus_count, epochs=10)

print("Nombre de mots du Vocabulaire (mots uniques du corpus), taille du vecteur : ", modele1.wv.vectors.shape)
print("Nombre de mots dans le corpus : ", modele1.corpus_total_words)
print("Nombre de documents : ", modele1.corpus_count)


In [ ]:
# On peut initialiser, construire le vocabulaire et entraîner le modèle au même temps
#modele1 = FastText(corpus_without_sw,vector_size=2,window=5,min_count=1)

In [ ]:
# Affichez les mots du vocabulaire et leurs vecteurs issus du modèle FastText
print("La taille du Vocabulaire: (mots uniques)", len(modele1.wv.key_to_index))
print("Vocabulaire:", modele1.wv.key_to_index)
print("Vecteur pour 'answer':", modele1.wv['answer'])

In [ ]:
# Affichage des mots du vocabulaire sur un plan sachant que chaque mot est plongé dans un vecteur de dimension 2

mots=list(modele1.wv.key_to_index.keys())
X=[modele1.wv[x][0] for x in mots ]
Y=[modele1.wv[x][1] for x in mots ]
plt.scatter(X, Y)
for i in range(len(X)):
    plt.annotate(mots[i],(X[i],Y[i]))
plt.show()

##Entrainer un modèle FastText avec notre corpus en supprimant les mots vides

In [ ]:
from gensim.models import FastText

# Initialisez le modèle avec les paramètres souhaités
modele = FastText(vector_size=2, window=5, min_count=1)

# Construisez le vocabulaire à partir du corpus
modele.build_vocab(corpus_without_sw)

# Entraînez le modèle
modele.train(corpus_without_sw, total_examples=modele.corpus_count, epochs=10)

print("Nombre de mots du Vocabulaire (mots uniques du corpus), taille du vecteur : ", modele.wv.vectors.shape)
print("Nombre de mots dans le corpus : ", modele.corpus_total_words)
print("Nombre de documents : ", modele.corpus_count)


In [ ]:
# On peut initialiser, construire le vocabulaire et entraîner le modèle au même temps
#modele = FastText(corpus_without_sw,vector_size=2,window=5,min_count=1)

In [ ]:
# Affichez les mots du vocabulaire et leurs vecteurs issus du modèle FastText
print("La taille du Vocabulaire: (mots uniques)", len(modele.wv.key_to_index))
print("Vocabulaire:", modele.wv.key_to_index)
print("Vecteur pour 'answer':", modele.wv['answer'])

In [ ]:
# Affichage des mots du vocabulaire sur un plan sachant que chaque mot est plongé dans un vecteur de dimension 2

mots=list(modele.wv.key_to_index.keys())
X=[modele.wv[x][0] for x in mots ]
Y=[modele.wv[x][1] for x in mots ]
plt.scatter(X, Y)
for i in range(len(X)):
    plt.annotate(mots[i],(X[i],Y[i]))
plt.show()

# Utiliser des modèles Fasttext pré-entainés
# Avec la librairie Gensim
Vous pouvez récupérer différents modèles fasttext pour l'anglais via cet url : https://fasttext.cc/docs/en/english-vectors.html

ou en fonction de la langue à

https://fasttext.cc/docs/en/crawl-vectors.html





In [ ]:
# télécharger par exemple le fichier  :
#https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
# déziper le fichier téléchargé => cc.en.300.vec
# charger le fichier dans votre environnement ou donner son path

In [ ]:
# Un exemple avec un fichier d'un modèle fasttext de la langue anglaise.
import requests
import zipfile
import os

url = "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip"
local_zip_path = "wiki-news-300d-1M.vec.zip"
# récupérer le modèle compressé

response = requests.get(url)
with open(local_zip_path, "wb") as f:
    f.write(response.content)

In [ ]:
#Extraction du fichier
with zipfile.ZipFile(local_zip_path, "r") as zip_ref:
    zip_ref.extractall("./")

# vérifier le contenu du répertoire courant après l'extraction
!ls

In [ ]:
from gensim.models import keyedvectors

model_path = "wiki-news-300d-1M.vec"
# utiliser la même fonction que pour le word2vec pour charger le modèle fasttext.

Plog_mots_fasttext = keyedvectors.load_word2vec_format(model_path, binary=False)



In [ ]:
# Exemple du vecteur caractéristique fasttext d'un mot présent dans le corpus initial
Plog_mots_fasttext['table']

In [ ]:
# calculer les vecteurs caractéristiques des phrases en moyennant les vecteurs fasttext des mots présents dans chaque phrase
Vec_Docs_fasttext = []
for doc in corpus:
    vec = doc_as_words_mean(doc,Plog_mots_fasttext)
    Vec_Docs_fasttext.append(vec)

matVec = np.array(Vec_Docs_fasttext)
print(matVec.shape)
print(matVec)

# Utiliser des modèles Fasttext pré-entainés

# Avec la librairie fasttext

In [ ]:
pip install fasttext

In [9]:
# Un exemple avec un fichier d'un modèle fasttext de la langue anglaise.
import requests
import zipfile
import os

url = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz"
#url = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz"
local_gz_path = "cc.en.300.bin.gz"


# récupérer le modèle compressé

response = requests.get(url)
with open(local_gz_path, "wb") as f:
    f.write(response.content)



In [10]:
#Extraction du fichier
import gzip
import shutil

output_file = "cc.en.300.bin"


with gzip.open(local_gz_path, 'rb') as local_gz_path:
  with open(output_file, 'wb') as out_file:
    shutil.copyfileobj(local_gz_path, out_file)


# vérifier le contenu du répertoire courant après l'extraction
!ls

cc.en.300.bin  cc.en.300.bin.gz  sample_data


In [ ]:
# Chargement du modèle pré entrainé
import fasttext

ft = fasttext.load_model('cc.en.300.bin')

len(ft.get_words())

In [ ]:
print(type(ft))

In [ ]:
dir(ft)

In [ ]:
ft.get_dimension()

In [ ]:
print(ft.get_word_vector("answer"))

In [8]:
def doc_as_words_mean_1(doc,mod):
    """
    Cette fonction calcule la moyenne des vecteurs des mots d'un document (phrase)
    """

    tailleVec = mod.get_dimension()

    vec_doc = np.zeros(tailleVec)
    nb_tokens = 0.0

    for token in doc:
      # Si le mot du document n'appartient au vocabulaire du modèle pré entainé, il n'est pas pris en considération
        try:
            vec_token = mod.get_word_vector(token)
            vec_doc = vec_doc + vec_token
            nb_tokens +=  1
        except:
            pass


    if (nb_tokens > 0.0):
        vec_doc = vec_doc/nb_tokens

    return vec_doc

In [ ]:
# calculer les vecteurs caractéristiques des phrases en moyennant les vecteurs fasttext des mots présents dans chaque phrase
Vec_Docs_fasttext = []
for doc in corpus:
    vec = doc_as_words_mean_1(doc,ft)
    Vec_Docs_fasttext.append(vec)

matVec = np.array(Vec_Docs_fasttext)
print(matVec.shape)
print(matVec)

# DOC2Vec
#C'est un modèle similaire au word2vec et permet de plonger un document dans un vecteur.


## Calcul du Doc2Vec sur le corpus en gardant les mots vides

In [7]:
# Associer à chaque document un label (tag), cet identifiant de document sera utilisé lors de l'apprentissage du modèle

from gensim.models.doc2vec import TaggedDocument

tagged_docs = [TaggedDocument(words=corpus[i],tags=["d"+str(i+1)]) for i in range(len(corpus))]

#premier doc par ex.
print(tagged_docs[0])

TaggedDocument<['if', 'there', 'is', 'anyone', 'out', 'there', 'who', 'still', 'doubt', 'that', 'america', 'is', 'a', 'place', 'where', 'all', 'thing', 'are', 'possible', 'who', 'still', 'wonder', 'if', 'the', 'dream', 'of', 'our', 'founder', 'is', 'alive', 'in', 'our', 'time', 'who', 'still', 'question', 'the', 'power', 'of', 'our', 'democracy', 'tonight', 'is', 'your', 'answer'], ['d1']>


In [16]:
from gensim.models.doc2vec import Doc2Vec

# Définition du modèle
modeleDoc = Doc2Vec(vector_size=2,window=5,min_count=1)

# Construction de dictionnaire dans un premier temps
modeleDoc.build_vocab(tagged_docs)

In [ ]:
# Affichage du vocabulaire du corpus
index=modeleDoc.wv.key_to_index.keys()
print(len(index))
print(index)

In [ ]:
# Entrainement du modèle
modeleDoc.train(tagged_docs,total_examples=modeleDoc.corpus_count,epochs=100)
print(modeleDoc.dv)

In [ ]:
# Les vecteurs caractéristiques des mots du vocabulaire
pd.DataFrame(modeleDoc.wv.vectors,columns=['V1','V2'],index=modeleDoc.wv.key_to_index.keys())

In [ ]:
# Les vecteurs caractéristiques des documents
dfDoc2Vec = pd.DataFrame(modeleDoc.dv.vectors,columns=['X1','X2'])
print(dfDoc2Vec)

In [ ]:
#Affichage des documents sachant qu'ils étaient représentés par des vecteurs de 2 dimensions
import seaborn as sns
sns.scatterplot(data=dfDoc2Vec,x='X1',y='X2')


## Calcul du Doc2Vec sur le corpus en gardant les mots vides




In [ ]:
# Associer à chaque document un label (tag), cet identifiant de document sera utilisé lors de l'apprentissage du modèle

from gensim.models.doc2vec import TaggedDocument

tagged_docs = [TaggedDocument(words=corpus_without_sw[i],tags=["d"+str(i+1)]) for i in range(len(corpus_without_sw))]

#premier doc par exemple
print(tagged_docs[0])

In [ ]:
# Affichage du deuxième document (phrase)
print(tagged_docs[1])

In [23]:
from gensim.models.doc2vec import Doc2Vec

# Définition du modèle
modeleDoc = Doc2Vec(vector_size=2,window=5,min_count=1)

#construction de dictionnaire dans un premier temps
modeleDoc.build_vocab(tagged_docs)

In [ ]:
#modélisation pour le positionnement des documents
modeleDoc.train(tagged_docs,total_examples=modeleDoc.corpus_count,epochs=100)
print(modeleDoc.dv)

In [ ]:
# Affichage du vocabulaire du corpus
index=modeleDoc.wv.key_to_index.keys()
index

In [ ]:
# Les vecteurs caractéristiques des mots du vocabulaire
pd.DataFrame(modeleDoc.wv.vectors,columns=['V1','V2'],index=modeleDoc.wv.key_to_index.keys())

In [ ]:
# Les vecteurs caractéristiques des documents
dfDoc2Vec = pd.DataFrame(modeleDoc.dv.vectors,columns=['X1','X2'])
print(dfDoc2Vec)

In [ ]:
#Affichage des documents sachant qu'ils étaient représentés par des vecteurs de 2 dimensions
import seaborn as sns
sns.scatterplot(data=dfDoc2Vec,x='X1',y='X2')